# Lab 4: Multi-Agent Workflow in Foundry (15 min)

## Objectives
- Create **agents in the Foundry Agent Service** (server-side, visible in the portal)
- Use **Connected Agents** to link agents together
- Create an **orchestrator** that delegates tasks sequentially to sub-agents
- View the **run steps** to follow the agent → agent delegation

## Concepts

### Multi-Agent Workflow in Foundry Agent Service

Foundry allows you to create **server-side multi-agent workflows** where:
- Each agent is created in the service (visible in the Foundry portal)
- An **orchestrator** uses `ConnectedAgentTool` to delegate tasks to sub-agents
- The service automatically manages communication and context between agents

### Sequential Workflow Pattern with Connected Agents
```
                         Foundry Agent Service
┌─────────────────────────────────────────────────────────────────┐
│                                                                 │
│   ┌──────────────┐                                              │
│   │ Orchestrator  │──── ConnectedAgentTool ──┐                  │
│   └──────┬───────┘                           │                  │
│          │                                   │                  │
│    ┌─────▼──────┐  ┌──────────┐  ┌──────────▼┐                 │
│    │Researcher  │→ │ Architect│→ │  Writer   │                  │
│    └────────────┘  └──────────┘  └───────────┘                  │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

The orchestrator decides the **order of calls** based on its instructions.

In [ ]:
# Setup — same pattern as Lab 2.1
from dotenv import load_dotenv
import os

load_dotenv("../.env")

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.ai.agents.models import ConnectedAgentTool, MessageTextContent, RunStepConnectedAgentToolCall
from azure.ai.agents.models import AgentThreadCreationOptions, ThreadMessageOptions
from azure.identity import DefaultAzureCredential

project_endpoint = os.getenv("AZURE_AI_AGENT_ENDPOINT")
model = os.getenv("MODEL_DEPLOYMENT", "gpt-4o")

# Project client — to create/manage agents (visible in the portal!)
project = AIProjectClient(
    endpoint=project_endpoint,
    credential=DefaultAzureCredential(),
)

# Agents client — for threads, runs and connected agents
agents_client = project.agents

print(f"✅ Project Client connected: {project_endpoint}")
print(f"   Model: {model}")

✅ Project Client conectado: https://foundry-mod2.services.ai.azure.com/api/projects/proj-tutor
   Modelo: gpt-4o


## 🖥️ Exercise 4.1: Sequential Pipeline with 3 Agents

Scenario: Process a technical proposal request with 3 **server-side** agents:

1. **Researcher** → Analyzes requirements and identifies technologies
2. **Architect** → Designs the technical solution
3. **Writer** → Creates the final formatted proposal

The **Orchestrator** uses `ConnectedAgentTool` to delegate sequentially.

In [ ]:
# Exercise 4.1: Create the 3 sub-agents in Foundry Agent Service
from azure.ai.agents import AgentsClient

agents = AgentsClient(
    endpoint=project_endpoint,
    credential=DefaultAzureCredential(),
)

# Agent 1: Researcher
researcher = agents.create_agent(
    model=model,
    name="researcher",
    instructions="""You are a technical analyst. Your task is to:
1. Analyze project requirements
2. Identify relevant Azure technologies
3. List risks and considerations
Respond in structured format with clear sections.""",
)
print(f"✅ Researcher created: {researcher.id}")

# Agent 2: Architect
architect = agents.create_agent(
    model=model,
    name="architect",
    instructions="""You are an Azure solutions architect. Your task is to:
1. Receive a technical analysis
2. Design an Azure architecture (services, integrations, flows)
3. Estimate sizing
Use ASCII diagrams when possible.""",
)
print(f"✅ Architect created: {architect.id}")

# Agent 3: Writer
writer = agents.create_agent(
    model=model,
    name="writer",
    instructions="""You are a technical writer. Your task is to:
1. Receive technical analysis and architecture
2. Create a professional and concise proposal
3. Include: Executive Summary, Proposed Solution, Timeline, Next Steps
Maximum 400 words.""",
)
print(f"✅ Writer created: {writer.id}")

# Create ConnectedAgentTool for each sub-agent
connected_tools = []
for agent in [researcher, architect, writer]:
    tool = ConnectedAgentTool(
        id=agent.id,
        name=agent.name,
        description=f"Specialized agent: {agent.name}",
    )
    connected_tools.extend(tool.definitions)

# Orchestrator agent — delegates sequentially to sub-agents
orchestrator = agents.create_agent(
    model=model,
    name="orchestrator",
    instructions="""You are a project manager. For each request, execute these steps in sequence:
1. First, ask the researcher to analyze the requirements
2. Then, ask the architect to design the solution based on the researcher's analysis
3. Finally, ask the writer to create the final proposal based on the analysis and architecture
Present the writer's final result to the user.""",
    tools=connected_tools,
)
print(f"✅ Orchestrator created: {orchestrator.id}")
print(f"   Connected agents: researcher, architect, writer")
print(f"\n💡 Go to the Foundry portal and verify that all 4 agents appear there!")

✅ Agente Investigador: investigador
✅ Agente Arquiteto: arquiteto
✅ Agente Redator: redator


In [ ]:
# Exercise 4.1: Execute the sequential workflow

request = """
Contoso wants to migrate their customer service system to the cloud.
Currently they use:
- On-premises SQL Server database with 500GB of data
- .NET Framework 4.8 web application
- Basic rule-based chatbot

Requirements:
- Intelligent AI chatbot (understand natural language questions)
- Search across product manuals (3000 PDF documents)
- Integration with existing CRM (Dynamics 365)
- High availability (99.9% SLA)
- GDPR compliance (data in Europe)
"""

print("=" * 60)
print("📋 CLIENT REQUEST")
print("=" * 60)
print(request)
print("⚙️ Executing workflow: orchestrator → researcher → architect → writer...")
print("   (Foundry Agent Service manages communication between agents)\n")

# Execute — create_thread_and_process_run does automatic polling
run = agents.create_thread_and_process_run(
    agent_id=orchestrator.id,
    thread=AgentThreadCreationOptions(
        messages=[ThreadMessageOptions(
            role="user",
            content=f"Analyze the following project requirements and create a proposal:\n{request}",
        )]
    ),
)
print(f"✅ Run completed! Status: {run.status}")

# Show run steps — each delegation to a sub-agent appears as a tool_call
steps = list(agents.run_steps.list(thread_id=run.thread_id, run_id=run.id))
print(f"\n📋 {len(steps)} Run Steps (execution order):")
for step in reversed(steps):
    print(f"  [{step.type}] {step.status}")
    if hasattr(step.step_details, "tool_calls"):
        for tc in step.step_details.tool_calls:
            if isinstance(tc, RunStepConnectedAgentToolCall):
                print(f"    → Delegated to agent: {tc.connected_agent.name}")

# Show final response
if run.status == "completed":
    messages = list(agents.messages.list(thread_id=run.thread_id))
    for msg in messages:
        if msg.role == "assistant":
            for content in msg.content:
                if isinstance(content, MessageTextContent):
                    print(f"\n{'=' * 60}")
                    print("📝 FINAL PROPOSAL (generated by the 3-agent workflow)")
                    print("=" * 60)
                    print(content.text.value)
            break

print("\n✅ Sequential 3-agent pipeline complete!")

✅ Workflow construído: investigador → arquiteto → redator

📋 PEDIDO DO CLIENTE

A empresa Contoso quer migrar o seu sistema de atendimento ao cliente para a cloud.
Atualmente usam:
- Base de dados SQL Server on-premises com 500GB de dados
- Aplicação web .NET Framework 4.8
- Chatbot básico baseado em regras

Requisitos:
- Chatbot inteligente com IA (entender perguntas em linguagem natural)
- Pesquisa nos manuais de produto (3000 documentos PDF)
- Integração com o CRM existente (Dynamics 365)
- Alta disponibilidade (99.9% SLA)
- RGPD compliance (dados na Europa)


⚙️ A executar pipeline de 3 agentes...

🔍 PASSO 1: Investigador
# Análise de Requisitos do Projeto

## 1. Requisitos
### Funcionais
1. **Chatbot inteligente com IA**: O sistema deve ter um chatbot com capacidades de Inteligência Artificial para compreender linguagem natural e interagir com os clientes de forma eficaz.
2. **Pesquisa nos manuais de produto**: Deve ser possível realizar pesquisas eficientes em textos contidos em 

## 🖥️ Exercise 4.2: Feedback Analysis Pipeline

Second workflow with 2 sub-agents + orchestrator:
1. **Classifier** → Categorizes and classifies sentiment
2. **Reporter** → Generates executive report

In [ ]:
# Exercise 4.2: Feedback analysis pipeline with 2 sub-agents

# Sub-agent: Classifier
classifier = agents.create_agent(
    model=model,
    name="classifier",
    instructions="""Classify each feedback with:
- sentiment: positive/negative/neutral
- category: UX, documentation, pricing, functionality
- priority: high/medium/low
Respond in valid JSON as an array of objects.""",
)
print(f"✅ Classifier: {classifier.id}")

# Sub-agent: Reporter
reporter = agents.create_agent(
    model=model,
    name="reporter",
    instructions="""Based on the feedback classification, generate a 5-line executive report with:
overall summary, strengths, areas for improvement, and a priority recommendation.""",
)
print(f"✅ Reporter: {reporter.id}")

# Feedback orchestrator
connected_fb = []
for agent in [classifier, reporter]:
    tool = ConnectedAgentTool(id=agent.id, name=agent.name, description=f"Agent: {agent.name}")
    connected_fb.extend(tool.definitions)

orq_feedback = agents.create_agent(
    model=model,
    name="orq_feedback",
    instructions="""You are a quality manager. For each set of feedbacks:
1. First, ask the classifier to categorize each feedback
2. Then, ask the reporter to generate the executive report based on the classification
Present the final report to the user.""",
    tools=connected_fb,
)
print(f"✅ Feedback orchestrator: {orq_feedback.id}")

feedbacks = [
    "The Foundry portal is very intuitive, I loved the model deployment experience!",
    "It took me 3 hours to configure authentication. The documentation is confusing.",
    "The pricing is reasonable for enterprises, but startups need a larger free tier.",
    "The code interpreter in agents is fantastic for data analysis.",
]

input_text = "\n".join([f"{i+1}. {f}" for i, f in enumerate(feedbacks)])

print("\n⚙️ Executing feedback pipeline...")
run_fb = agents.create_thread_and_process_run(
    agent_id=orq_feedback.id,
    thread=AgentThreadCreationOptions(
        messages=[ThreadMessageOptions(role="user", content=f"Analyze these feedbacks:\n{input_text}")]
    ),
)
print(f"✅ Run completed! Status: {run_fb.status}")

# Run steps
steps_fb = list(agents.run_steps.list(thread_id=run_fb.thread_id, run_id=run_fb.id))
print(f"\n📋 {len(steps_fb)} Run Steps:")
for step in reversed(steps_fb):
    if hasattr(step.step_details, "tool_calls"):
        for tc in step.step_details.tool_calls:
            if isinstance(tc, RunStepConnectedAgentToolCall):
                print(f"  → Delegated to agent: {tc.connected_agent.name}")

# Final response
if run_fb.status == "completed":
    messages_fb = list(agents.messages.list(thread_id=run_fb.thread_id))
    for msg in messages_fb:
        if msg.role == "assistant":
            for content in msg.content:
                if isinstance(content, MessageTextContent):
                    print(f"\n{'=' * 60}")
                    print("📊 FEEDBACK REPORT")
                    print("=" * 60)
                    print(content.text.value)
            break

print("\n✅ Feedback pipeline complete!")

In [ ]:
# Cleanup — delete all agents created in Foundry
for agent_id in [orchestrator.id, researcher.id, architect.id, writer.id,
                 orq_feedback.id, classifier.id, reporter.id]:
    try:
        agents.delete_agent(agent_id)
        print(f"🗑️ Agent deleted: {agent_id}")
    except Exception as e:
        print(f"⚠️ Error: {e}")

print("\n✅ Cleanup complete! Check the Foundry portal to verify the agents are gone.")

## 🧪 Step 2: Declarative Workflow Agent in the Portal (Preview)

### What is a Workflow Agent?

Beyond the **ConnectedAgentTool** approach (used above), Foundry introduces the concept of **Workflow Agent** — a declarative agent whose behavior is defined by a **CSDL YAML** file (Conversational State Description Language).

With this approach:
- The **workflow is a resource** within Foundry (visible in the portal as a "Workflow" type agent)
- The orchestration logic is **declarative** — it doesn't depend on natural language instructions
- Foundry manages execution, checkpoints, and conversation state automatically

### Comparison: ConnectedAgentTool vs Workflow Agent

| Aspect | ConnectedAgentTool (above) | Workflow Agent (CSDL) |
|--------|---------------------------|----------------------|
| **Orchestration** | Orchestrator instructions (LLM decides) | Declarative (YAML defines the sequence) |
| **Control** | Probabilistic (depends on the model) | Deterministic (fixed steps) |
| **Creation** | Code (`create_agent` + tools) | Portal / YAML |
| **State** | Thread managed by the agent | Automatic checkpoints |
| **Maturity** | ✅ GA (production) | 🧪 Preview |

---

### 📋 Step by step: Create a Workflow Agent in the Azure AI Foundry Portal

#### Prerequisite
The sub-agents (**researcher**, **architect**, **writer**) must already exist in the project — we created them in Exercise 4.1 above.

#### 1. Access the portal
- Go to [https://ai.azure.com](https://ai.azure.com)
- Select your project (e.g., `proj-tutor`)

#### 2. Navigate to Agents
- In the top menu, click **Build** → in the left pane select **Agents**
- You'll see the sub-agents already created (researcher, architect, writer)

#### 3. Create a new Workflow type agent
- Click **"+ New agent"**
- Select the **"Workflow"** type (instead of "Prompt")
- Name it: `workflow_proposal`

#### 4. Define the CSDL YAML
In the workflow editor, paste the following definition:

```yaml
kind: Workflow
triggers:
  - kind: UserMessage
actions:
  - kind: InvokeAzureAgent
    agent: investigador
  - kind: InvokeAzureAgent
    agent: arquiteto
  - kind: InvokeAzureAgent
    agent: redator
```

**What each block does:**
| Block | Description |
|-------|-------------|
| `kind: Workflow` | Declares that this agent is a workflow |
| `triggers: UserMessage` | The workflow starts when the user sends a message |
| `actions: InvokeAzureAgent` | Each action invokes a sub-agent in the defined order |

#### 5. Publish and test
- Click **"Save"** / **"Publish"**
- In the test panel (chat), send the request:

> *"Contoso wants to migrate their customer service system to the cloud. They use SQL Server 500GB, .NET Framework 4.8, basic chatbot. They need an AI chatbot, search across 3000 PDFs, Dynamics 365 integration, 99.9% SLA, GDPR compliance."*

- Foundry will automatically execute: **researcher → architect → writer**
- Each step appears in the execution panel with its outputs

#### 6. Verify the execution
In the chat side panel, you can see:
- ✅ Each action executed (InvokeAzureAgent)
- 📄 The output from each sub-agent
- ⏱️ Execution time per step
- 🔄 Automatic checkpoints

> ⚠️ **Note:** Workflow Agent is in **Preview** (requires feature flag `WorkflowAgents=V1Preview`). The portal interface may vary depending on the version available in your tenant.

## ✅ Summary

In this lab you learned:
- How to create **server-side agents** in the Foundry Agent Service with `AgentsClient.create_agent()`
- How to use **`ConnectedAgentTool`** to link sub-agents to an orchestrator
- The **sequential multi-agent workflow** pattern: orchestrator delegates tasks in sequence
- How to inspect **run steps** to see agent → agent delegation
- How to clean up agents with `delete_agent()`
- How to create a **declarative Workflow Agent** with `WorkflowAgentDefinition` and CSDL YAML (Preview)

### Multi-Agent Workflow Approaches in Foundry

| Approach | Technique | Maturity | When to use |
|----------|-----------|----------|-------------|
| **ConnectedAgentTool** | Orchestrator + sub-agents via tools | ✅ GA | Flexible workflows, adaptive logic |
| **Workflow Agent (CSDL)** | Declarative definition in YAML | 🧪 Preview | Deterministic workflows, fixed pipelines |

### Orchestration Patterns
| Pattern | How it works | When to use |
|---------|--------------|-------------|
| **Sequential** | Orchestrator calls agents in order per instructions | Linear pipelines (this lab) |
| **Parallel** | Orchestrator calls multiple agents in the same step | Independent tasks |
| **Hierarchical** | Orchestrator delegates to sub-orchestrators | Complex workflows |

### Lab 2.1 vs Lab 4
| | Lab 2.1 | Lab 4 |
|---|---------|--------|
| **Agents** | 1 individual agent | 4+ coordinated agents |
| **Coordination** | Manual (code) | Automatic (ConnectedAgentTool / CSDL) |
| **Context** | `previous_response_id` | Managed by the service |

**Next:** [Lab 5 - Knowledge / RAG](../lab05/README.md)